In [1]:
import pandas as pd
df = pd.read_csv(r"C:\Users\appuv\Desktop\internship\Cleaned Hotel.csv")

In [2]:
from sqlalchemy import create_engine
import pandas as pd

engine = create_engine('sqlite:///:memory:')
df.to_sql('hotel_bookings', engine, index=False, if_exists='replace')

print("Connection created")

Connection created


In [3]:
import pandas as pd

query = """
SELECT hotel, COUNT(*) AS total_bookings
FROM hotel_bookings
GROUP BY hotel
ORDER BY total_bookings DESC
LIMIT 5;
"""

pd.read_sql(query, engine)

,hotel,total_bookings
0,City Hotel,52349
1,Resort Hotel,33265


In [4]:
# Overall KPIs

query = """
SELECT 
    COUNT(*) AS total_bookings,
    SUM(is_canceled) AS canceled_bookings,
    ROUND(100.0 * SUM(is_canceled) / COUNT(*), 2) AS cancel_rate_pct,
    ROUND(AVG(lead_time), 1) AS avg_lead_time_days,
    ROUND(SUM(adr * (stays_in_weekend_nights + stays_in_week_nights)), 2) AS total_revenue
FROM hotel_bookings;
"""
pd.read_sql(query, engine)

,total_bookings,canceled_bookings,cancel_rate_pct,avg_lead_time_days,total_revenue
0,85614,23839,27.84,80.7,34454585.11


In [5]:
#Cancellation rate by hotel type

query = """
SELECT 
    hotel,
    COUNT(*) AS bookings,
    ROUND(100.0 * SUM(is_canceled) / COUNT(*), 2) AS cancel_rate_pct
FROM hotel_bookings
GROUP BY hotel
ORDER BY cancel_rate_pct DESC;
"""
pd.read_sql(query, engine)

,hotel,bookings,cancel_rate_pct
0,City Hotel,52349,30.45
1,Resort Hotel,33265,23.75


In [6]:
# Peak months for bookings and cancellations

query = """
SELECT 
    arrival_date_month,
    COUNT(*) AS total_bookings,
    SUM(is_canceled) AS canceled,
    ROUND(100.0 * SUM(is_canceled) / COUNT(*), 2) AS cancel_rate_pct
FROM hotel_bookings
GROUP BY arrival_date_month
ORDER BY 
    CASE arrival_date_month
        WHEN 'January' THEN 1 WHEN 'February' THEN 2 WHEN 'March' THEN 3 WHEN 'April' THEN 4
        WHEN 'May' THEN 5 WHEN 'June' THEN 6 WHEN 'July' THEN 7 WHEN 'August' THEN 8
        WHEN 'September' THEN 9 WHEN 'October' THEN 10 WHEN 'November' THEN 11 WHEN 'December' THEN 12
    END;
"""
pd.read_sql(query, engine)

,arrival_date_month,total_bookings,canceled,cancel_rate_pct
0,January,4573,1026,22.44
1,February,5957,1395,23.42
2,March,7369,1818,24.67
3,April,7786,2394,30.75
4,May,8211,2428,29.57
5,June,7652,2344,30.63
6,July,9901,3183,32.15
7,August,11103,3614,32.55
8,September,6541,1619,24.75
9,October,6728,1610,23.93


In [7]:
#Revenue by market segment

query = """
SELECT 
    market_segment,
    COUNT(*) AS bookings,
    ROUND(SUM(adr * (stays_in_weekend_nights + stays_in_week_nights)), 2) AS revenue
FROM hotel_bookings
WHERE is_canceled = 0
GROUP BY market_segment
ORDER BY revenue DESC
LIMIT 5;
"""
pd.read_sql(query, engine)

,market_segment,bookings,revenue
0,Online TA,33040,12973686.94
1,Offline TA/TO,11596,4595208.77
2,Direct,9866,3927104.69
3,Groups,3406,922928.01
4,Corporate,3632,483240.83


In [8]:
#Lead time impact on cancellations

query = """
SELECT 
    CASE 
        WHEN lead_time <= 30 THEN '0-30 days'
        WHEN lead_time <= 90 THEN '31-90 days'
        WHEN lead_time <= 180 THEN '91-180 days'
        ELSE '180+ days'
    END AS lead_time_bucket,
    COUNT(*) AS bookings,
    ROUND(100.0 * AVG(is_canceled), 2) AS cancel_rate_pct
FROM hotel_bookings
GROUP BY lead_time_bucket
ORDER BY MIN(lead_time);
"""
pd.read_sql(query, engine)

,lead_time_bucket,bookings,cancel_rate_pct
0,0-30 days,33333,16.71
1,31-90 days,22519,32.21
2,91-180 days,18127,35.12
3,180+ days,11635,39.95


In [9]:
#Top 5 countries by bookings

query = """
SELECT 
    country,
    COUNT(*) AS bookings
FROM hotel_bookings
WHERE country IS NOT NULL
GROUP BY country
ORDER BY bookings DESC
LIMIT 5;
"""
pd.read_sql(query, engine)

,country,bookings
0,PRT,26037
1,GBR,10363
2,FRA,8786
3,ESP,7182
4,DEU,5366


In [10]:
# Bottom 5 countries by bookings

query = """
SELECT 
    country,
    COUNT(*) AS bookings
FROM hotel_bookings
WHERE country IS NOT NULL
GROUP BY country
ORDER BY bookings ASC
LIMIT 5;
"""
pd.read_sql(query, engine)

,country,bookings
0,AIA,1
1,ASM,1
2,ATF,1
3,BDI,1
4,BFA,1
